# Hikrobot GigE (PoE) — bench test

Interactive check of the `HikRobot` driver against the camera at **10.0.1.50**.

**Prerequisites (Windows):**
- Hikrobot **MVS** installed — the driver finds the bindings/DLL automatically, even in a process launched before the install
- The camera reachable from this PC: **NIC on the same subnet** (e.g. host `10.0.1.x/24`), MVS client closed (GigE control is **exclusive** — the MVS viewer holding the camera blocks us)
- `numpy`, `opencv-python`, `matplotlib` in this Python environment

⚠️ **Cross-subnet does not work for open** (verified on this bench): with the PC on `10.0.3.x` Wi-Fi and the camera on `10.0.1.x`, ping and the accessibility query pass through the router, but `connect` fails with `MV_E_UDP_RECV_DATA` — the open handshake needs the *camera* to initiate flows back to the PC, and stateful routers drop those. Fix: plug this PC into the camera's PoE switch (NIC gets a `10.0.1.x` address), or run the same test from the Pi, which already shares the camera's subnet.

In [1]:
# Run from the example/ folder — put the repo root on the path
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from camera import HikRobot

CAMERA_IP = "10.0.1.50"

## 1. Discover cameras on the network
Empty list = SDK missing **or** no camera answering discovery (check PoE link / subnet / firewall — GigE discovery is UDP broadcast).

In [2]:
HikRobot.all_device()

[]

## 2. Connect
`raise_on_fail=True` so a failure shows the actionable message instead of a silent `False`.

In [3]:
cam = HikRobot()
cam.connect(ip=CAMERA_IP, raise_on_fail=True)
print("serial:       ", cam.serial_number)
print("ip:           ", cam.ip)
print("stream_actual:", cam.stream_actual)
print("state:        ", cam.state)

RuntimeError: Hikrobot MVS runtime not available (Could not find module 'MvCameraControl.dll' (or one of its dependencies). Try using the full path with constructor syntax.) — install MVS from hikrobotics.com (it ships the Python bindings under C:\Program Files (x86)\MVS\Development\Samples\Python\MvImport), then restart this process/kernel. Alternatively vendor the MvImport folder into <repo>/mvs/MvImport — see mvs/README.md

## 3. Grab a frame and show it
Same 9-tuple as the D405 — depth/ir slots come back `None` on this color-only device.

In [ ]:
import cv2
import matplotlib.pyplot as plt

_, _, _, _, _, color_img, depth_int, _, ts = cam.get_all()
print("frame:", color_img.shape, "dtype:", color_img.dtype)

plt.figure(figsize=(10, 7))
plt.imshow(cv2.cvtColor(color_img, cv2.COLOR_BGR2RGB))
plt.title(f"{cam.serial_number} @ {cam.ip}")
plt.axis("off")
plt.show()

## 4. Intrinsics
Without authored `K`/`D` this is the **nominal placeholder** (focal = image width, centered pp) — fine for drawing, not for metric work.

In [ ]:
print("source:", cam.intr_source)
print("K:", cam.get_K())
print("D:", cam.get_D())

## 5. Exposure & white balance
Real knobs on this sensor. Exposure is in **µs** (same unit as the D405).

In [ ]:
print("auto exposure now:", cam.get_exposure(), "us")

# pin a manual exposure, grab, compare — then back to auto
print("pinned:", cam.set_exposure(10000), "us")   # 10 ms
_, _, _, _, _, img_manual, _, _, _ = cam.get_all()
print("back to auto:", cam.auto_exposure(True), "us")

plt.figure(figsize=(10, 7))
plt.imshow(cv2.cvtColor(img_manual, cv2.COLOR_BGR2RGB))
plt.title("manual exposure 10 ms")
plt.axis("off")
plt.show()

In [ ]:
# white balance: let auto settle on the lit scene, then hold (deterministic color)
cam.white_balance({"auto": True})
# ... later, once colors look right:
# cam.white_balance({"hold": True})

## 6. Throughput check
A few timed grabs — over PoE expect grab time roughly `1/fps + conversion`; the SDK converts Bayer→BGR on the host.

In [ ]:
import time

N = 10
t0 = time.time()
for _ in range(N):
    cam.get_all()
dt = time.time() - t0
print(f"{N} grabs in {dt:.2f}s  ->  {N/dt:.1f} fps  ({1000*dt/N:.0f} ms/grab)")

## 7. Close
Release the exclusive GigE control channel — the MVS viewer (or the Pi) can't open the camera while this notebook holds it.

In [ ]:
cam.close()